Thursday: one table, and the honest answer about tools

> "One table, one row per customer, refreshed on Monday. How recently they bought, how often,
> how much, their segment, whether the monsoon sale reached them, and last week's flags."

> "The warehouse queries are fine for Finance, but Marketing's analysts live in Python. Build
> them the table in pandas, from the warehouse, and make it refreshable in one run."

And from a senior analyst, the question the week has been building to:

> "You did the tree in plain Python in Week 1, in SQL on Monday. Do it a third way now, and tell
> me honestly which tool you would pick for which job."

**MAP** split apply combine  ->  **DO** the table  ->  **SEE** the merge refuse  ->
**CHECK** the reshape  ->  **SUM** the tool note.

In [1]:
import pathlib
import sys

import pandas as pd

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

eng = kit.engine()
orders = pd.read_sql("SELECT * FROM orders", eng)
customers = pd.read_sql("SELECT * FROM customers", eng)
exposure = pd.read_sql("SELECT * FROM campaign_exposure", eng)
print(f"pandas {pd.__version__}")
print(f"orders {len(orders):,}  customers {len(customers):,}  exposure {len(exposure):,}")

pandas 3.0.5
orders 1,000  customers 340  exposure 136


## MAP. groupby is the Week 1 accumulator, automated

Week 1 wrote the splitting and the combining by hand and kept a dictionary. `groupby` does the
splitting and the combining and leaves you the applying, which is the only part that was ever
about your question.

In [2]:
kit.flow(["1,000 order rows", "split by customer", "apply sum, count, max", "one row each"],
         lit=[1], title="Split, apply, combine")

# The Week 1 version, kept for comparison
by_hand = {}
for row in orders.itertuples():
    by_hand[row.customer_id] = by_hand.get(row.customer_id, 0) + float(row.amount)
print(f"by hand: {len(by_hand)} customers")

one_line = orders.groupby("customer_id")["amount"].sum()
print(f"one line: {len(one_line)} customers")
kit.check("the two agree on every customer",
          all(abs(by_hand[c] - float(one_line[c])) < 0.01 for c in by_hand))

by hand: 301 customers
one line: 301 customers


## DO. Named aggregations, so the table can be audited

`orders.groupby("customer_id").sum()` also runs. It sums every numeric column, including ones
nobody meant to add up, and names them after the inputs. Six months later nobody can tell
whether a column was chosen or inherited.

In [3]:
AS_OF = pd.Timestamp("2026-09-30")

table = orders.groupby("customer_id").agg(
    frequency=("order_id", "count"),
    monetary=("amount", "sum"),
    last_order=("order_date", "max"),
).reset_index()
table["recency_days"] = (AS_OF - pd.to_datetime(table["last_order"])).dt.days
kit.check("one row per customer who ordered", table["customer_id"].is_unique)
kit.check("recency is counted from a stated date, not from today",
          table["recency_days"].min() >= 0, f"min {table['recency_days'].min()} days")
table.head(6)

,customer_id,frequency,monetary,last_order,recency_days
0,C-0001,2,2840.0,2026-08-06,55
1,C-0002,4,6700.0,2026-07-05,87
2,C-0003,6,9710.0,2026-08-15,46
3,C-0004,2,3120.0,2026-08-11,50
4,C-0005,3,4970.0,2026-09-19,11
5,C-0006,4,9630.0,2026-09-11,19


### Why a reference date rather than today

Counting from `Timestamp.now()` makes the table change meaning every time it is rebuilt, so two
people running the same code on two days get different recency and neither is wrong. State the
date, and the table means the same thing whenever it runs.

## SEE. The merge that refuses

A merge is a join with a different name, and it carries Tuesday's danger. The exposure feed is
the many side.

In [4]:
loose = table.merge(exposure, on="customer_id", how="left")
print(f"rows in : {len(table)}")
print(f"rows out: {len(loose)}")
print(f"gained  : {len(loose) - len(table)}")
kit.check("the merge grew the table", len(loose) > len(table))

rows in : 301
rows out: 307
gained  : 6


In [5]:
try:
    table.merge(exposure, on="customer_id", how="left", validate="one_to_one")
except Exception as e:
    print(type(e).__module__ + "." + type(e).__name__)
    print(str(e))

pandas.errors.MergeError
Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
 customer_id
     C-0001
     C-0002
     C-0003
     C-0006
     C-0007 ...


### The loud version of Tuesday's count check

Tuesday you took the row count yourself and compared it. `validate=` makes the library take it
for you and refuse to continue. The check has not changed; forgetting it has become impossible
rather than merely unwise.

Fixing it is a decision rather than a keyword.

In [6]:
kit.tree({"label": "6 duplicate keys", "branches": [
    ("drop_duplicates", {"label": "keeps a row, picks which arbitrarily"}),
    ("aggregate first", {"label": "keeps both facts"}),
    ("ask the feed owner", {"label": "fixes it upstream"})]},
    taken=["aggregate first"], title="Three ways out, and they are not equal")

seen = (exposure.groupby("customer_id")
        .agg(exposed=("campaign_id", "count"),
             first_seen=("exposed_date", "min")).reset_index())
table = table.merge(seen, on="customer_id", how="left", validate="one_to_one")
table["exposed"] = table["exposed"].fillna(0).astype(int)
kit.check("the validated merge kept the row count", len(table) == len(one_line),
          f"{len(table)} rows")
kit.check("every customer has an exposure count, zero included",
          table["exposed"].notna().all())

## DO. Segment, and the table the growth team asked for

In [7]:
table = table.merge(customers[["customer_id", "segment", "city"]],
                    on="customer_id", how="left", validate="one_to_one")
wanted = ["customer_id", "frequency", "monetary", "last_order", "recency_days",
          "exposed", "first_seen", "segment", "city"]
kit.check("every column the growth team asked for is present",
          list(table.columns) == wanted, f"{table.shape[1]} columns: {list(table.columns)}")
kit.sql_table("SELECT segment, count(*) AS customers FROM customers GROUP BY segment "
              "ORDER BY segment", caption="The segments the table now carries")
table.head(6)

segment,customers
Business,40
Retail-Core,150
Retail-Plus,120
Student,30


,customer_id,frequency,monetary,last_order,recency_days,exposed,first_seen,segment,city
0,C-0001,2,2840.0,2026-08-06,55,2,2026-08-03,Retail-Core,Delhi
1,C-0002,4,6700.0,2026-07-05,87,2,2026-08-03,Retail-Core,Chennai
2,C-0003,6,9710.0,2026-08-15,46,2,2026-08-03,Retail-Core,Delhi
3,C-0004,2,3120.0,2026-08-11,50,0,NaN,Retail-Core,Bengaluru
4,C-0005,3,4970.0,2026-09-19,11,0,NaN,Retail-Core,Bengaluru
5,C-0006,4,9630.0,2026-09-11,19,2,2026-08-03,Retail-Core,Hyderabad


## CHECK. A reshape changes the question a table answers

Months down the page answers how this moved. Months across the page answers how these compare.
Nothing is added or removed; the shape decides which question is easy to ask.

In [8]:
q2 = orders[orders["quarter"] == "Q2"].copy()
q2["month"] = pd.to_datetime(q2["order_date"]).dt.strftime("%b")
monthly = q2.groupby(["customer_id", "month"], as_index=False)["amount"].sum()

wide = monthly.pivot_table(index="customer_id", columns="month",
                           values="amount", aggfunc="sum")
print("long:", monthly.shape, " wide:", wide.shape)
kit.check("widening loses no customer", wide.shape[0] == monthly["customer_id"].nunique())
kit.matrix(["long", "wide"], ["one row is", "easy question"],
           [["a customer-month", "how did this move"],
            ["a customer", "how do these compare"]],
           title="Same numbers, two shapes")

long: (359, 3)  wide: (227, 3)


### The wrong index runs perfectly and means nothing

Index by `order_id` instead of `customer_id` and you get one row per order with a column per
month, almost all of them empty. Read the row labels aloud before reading any value: "each row
is one order" is the moment it becomes obvious, and no total check would have found it.

In [9]:
wrong = q2.pivot_table(index="order_id", columns="month", values="amount", aggfunc="sum")
filled = wrong.notna().sum().sum() / (wrong.shape[0] * wrong.shape[1])
print(f"rows: {wrong.shape[0]}, columns: {wrong.shape[1]}, cells filled: {filled:.1%}")
kit.check("the wrong index leaves the grid almost empty", filled < 0.4, f"{filled:.1%}")

rows: 462, columns: 3, cells filled: 33.3%


## SUM. The same question, three tools

Revenue per segment, three ways. All three are correct and they are not interchangeable.

In [10]:
merged = orders.merge(customers[["customer_id", "segment"]], on="customer_id",
                      how="left", validate="many_to_one")
pandas_way = merged.groupby("segment")["amount"].sum().sort_values(ascending=False)
sql_way = kit.sql("""SELECT c.segment, sum(o.amount) AS amount FROM orders o
                     JOIN customers c USING (customer_id) GROUP BY c.segment""")
sql_map = {r["segment"]: float(r["amount"]) for r in sql_way}
agree = all(abs(float(pandas_way[s]) - sql_map[s]) < 0.01 for s in sql_map)
kit.check("SQL and pandas return the same numbers", agree, str(dict(sql_map)))

kit.matrix(["plain Python", "SQL", "pandas"],
           ["owns", "never"],
           [["what you must explain line by line", "anything at scale"],
            ["numbers Finance acts on", "exploration, iteration is slow"],
            ["the analyst's own iteration", "the source of truth"]],
           title="The operating rule, which is the day's deliverable")
kit.check_summary()